<a href="https://colab.research.google.com/github/Firefox-097/Steganography/blob/main/Steganography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install kaggle


!mkdir ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json  ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!kaggle datasets download -d marcozuppelli/stegoimagesdataset
!unzip -q /content/stegoimagesdataset.zip -d /content/stegoimages

Dataset URL: https://www.kaggle.com/datasets/marcozuppelli/stegoimagesdataset
License(s): DbCL-1.0
100% 1.51G/1.51G [01:29<00:00, 18.1MB/s]



In [17]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Point to the directory containing the class subdirectories ('clean', 'stego')
base_dir = "/content/stegoimages/test/test"

# Create a data generator with a validation split
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary',
    subset='training' # Specify training subset
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary',
    subset='validation' # Specify validation subset
)


Found 4872 images belonging to 2 classes.
Found 1217 images belonging to 2 classes.


In [16]:
!ls -R /content/stegoimages

/content/stegoimages:
dataset_information.csv  test

/content/stegoimages/test:
test

/content/stegoimages/test/test:
clean  stego

/content/stegoimages/test/test/clean:
04001.png  04287.png  04573.png  04859.png  05145.png  05431.png  05717.png
04002.png  04288.png  04574.png  04860.png  05146.png  05432.png  05718.png
04003.png  04289.png  04575.png  04861.png  05147.png  05433.png  05719.png
04004.png  04290.png  04576.png  04862.png  05148.png  05434.png  05720.png
04005.png  04291.png  04577.png  04863.png  05149.png  05435.png  05721.png
04006.png  04292.png  04578.png  04864.png  05150.png  05436.png  05722.png
04007.png  04293.png  04579.png  04865.png  05151.png  05437.png  05723.png
04008.png  04294.png  04580.png  04866.png  05152.png  05438.png  05724.png
04009.png  04295.png  04581.png  04867.png  05153.png  05439.png  05725.png
04010.png  04296.png  04582.png  04868.png  05154.png  05440.png  05726.png
04011.png  04297.png  04583.png  04869.png  05155.png  05441.png  0572

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(256, 256, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,839,105 (56.61 MB)

 Trainable params: 14,839,105 (56.61 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
from PIL import Image
import os

def check_image_files(generator, dataset_name):
    print(f"Checking {dataset_name} images...")
    problem_files = []
    for i in range(generator.n):
        file_path = os.path.join(generator.directory, generator.filenames[i])
        try:
            img = Image.open(file_path)
            img.verify() # Verify that it is an image
            img.close()
        except Exception as e:
            problem_files.append((file_path, str(e)))
            print(f"Problematic file in {dataset_name}: {file_path} - Error: {e}")

    if not problem_files:
        print(f"No problematic files found in {dataset_name}.")
    else:
        print(f"Found {len(problem_files)} problematic files in {dataset_name}.")
    return problem_files

# Check train_gen files
problematic_train_files = check_image_files(train_gen, 'training')

# Check val_gen files
problematic_val_files = check_image_files(val_gen, 'validation')


Checking training images...
Problematic file in training: /content/stegoimages/test/test/stego/image_05363_url_0.png - Error: cannot identify image file '/content/stegoimages/test/test/stego/image_05363_url_0.png'
Found 1 problematic files in training.
Checking validation images...
No problematic files found in validation.


In [24]:
import os

# Remove the identified problematic file
for file_path, _ in problematic_train_files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Removed problematic file: {file_path}")
    else:
        print(f"File not found (already removed or wrong path): {file_path}")

# Re-run the ImageDataGenerator setup to ensure it re-scans the directory
# and updates the generators after file removal. This is a crucial step.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_dir = "/content/stegoimages/test/test"
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print("Data generators re-initialized after removing problematic files.")

Removed problematic file: /content/stegoimages/test/test/stego/image_05363_url_0.png
Found 4871 images belonging to 2 classes.
Found 1217 images belonging to 2 classes.
Data generators re-initialized after removing problematic files.


In [ ]:
model.save("/content/drive/MyDrive/steganography_model.keras")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])

Final Training Accuracy: 0.75
Final Validation Accuracy: 0.75


In [ ]:

from google.colab import files
import shutil

uploaded = files.upload()
original_filename = next(iter(uploaded))
shutil.move(original_filename, "/content/test.jpeg")

from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/test.jpeg"

img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)



Saving Screenshot 2026-04-20 223409.png to Screenshot 2026-04-20 223409.png


In [ ]:
prediction = model.predict(img_array)
print("Prediction Score:", prediction[0][0])
if prediction[0][0] > 0.5:
    print("Stego Image")
    secret=True
else:
    print("Cover Image")
    secret=False


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction Score: 0.74844897
Stego Image


In [ ]:
!pip install stegano

from stegano import lsb

if secret:
    message = lsb.reveal(img_path)

    if message:
        print("Hidden Message:", message)
    else:
        print("No hidden message found")

IndexError: Impossible to detect message.

In [ ]:
from stegano import lsb

secret_message = input("Enter secret message: ")

secret_image = lsb.hide(img_path, secret_message)

secret_image.save("encoded.png")

print("Encoded image saved as encoded.png")

In [ ]:
print(train_gen.class_indices)

{'clean': 0, 'stego': 1}


In [ ]:
import numpy as np

labels = train_gen.classes

unique, counts = np.unique(labels, return_counts=True)

print(dict(zip(unique, counts)))


{np.int32(0): np.int64(4000), np.int32(1): np.int64(12000)}


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/steganography_model.keras")

print("Model loaded successfully!")

Model loaded successfully!


In [26]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(256,256,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/tmp/ipykernel_537/2343777330.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 128, 128,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 128, 128,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 128, 128,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 128, 128,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 128, 128,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 128, 128,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 129, 129,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 64, 64,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 64,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 64,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 64, 64,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [27]:
history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)

Epoch 1/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 86s 432ms/step - accuracy: 0.6567 - loss: 0.6567 - val_accuracy: 0.6713 - val_loss: 0.6484
Epoch 2/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 36s 233ms/step - accuracy: 0.6734 - loss: 0.6260 - val_accuracy: 0.6713 - val_loss: 0.6802
Epoch 3/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 32s 208ms/step - accuracy: 0.6773 - loss: 0.6168 - val_accuracy: 0.6713 - val_loss: 0.6753
Epoch 4/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 32s 211ms/step - accuracy: 0.6822 - loss: 0.6100 - val_accuracy: 0.6713 - val_loss: 0.7039
Epoch 5/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 33s 218ms/step - accuracy: 0.6863 - loss: 0.6027 - val_accuracy: 0.6680 - val_loss: 0.7530
Epoch 6/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 32s 206ms/step - accuracy: 0.6966 - loss: 0.5936 - val_accuracy: 0.6721 - val_loss: 0.7827
Epoch 7/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 32s 208ms/step - accuracy: 0.6953 - loss: 0.5898 - val_accuracy: 0.6615 - val_loss: 0.7413
Epoch 8/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 32s 212ms/step - accuracy: 0.7109 - loss: 0

In [28]:
print(model.summary())

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 128, 128,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 128, 128,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 128, 128,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 128, 128,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 128, 128,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 128, 128,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 129, 129,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 64, 64,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 64,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 64,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 64, 64,    │      2,304 │ block_1_depthwis

 Total params: 2,750,277 (10.49 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 328,196 (1.25 MB)

None


In [29]:
for layer in model.layers[:10]:
    print(layer.name)

input_layer_1
Conv1
bn_Conv1
Conv1_relu
expanded_conv_depthwise
expanded_conv_depthwise_BN
expanded_conv_depthwise_relu
expanded_conv_project
expanded_conv_project_BN
block_1_expand


In [30]:
print(train_gen.class_indices)

import numpy as np

labels = train_gen.classes
unique, counts = np.unique(labels, return_counts=True)

print(dict(zip(unique, counts)))

{'clean': 0, 'stego': 1}
{np.int32(0): np.int64(1600), np.int32(1): np.int64(3271)}


In [31]:
import os
import random

stego_dir = "/content/stegoimages/test/test/stego"

files = os.listdir(stego_dir)

random.shuffle(files)

remove_count = len(files) - 1600

for f in files[:remove_count]:
    os.remove(os.path.join(stego_dir, f))

print("Removed", remove_count, "stego images")

Removed 2488 stego images


In [32]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_dir = "/content/stegoimages/test/test"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 2880 images belonging to 2 classes.
Found 720 images belonging to 2 classes.


In [33]:
import numpy as np

labels = train_gen.classes
unique, counts = np.unique(labels, return_counts=True)

print(dict(zip(unique, counts)))

{np.int32(0): np.int64(1600), np.int32(1): np.int64(1280)}


In [34]:
history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 33s 368ms/step - accuracy: 0.6135 - loss: 0.6190 - val_accuracy: 0.4319 - val_loss: 1.1621
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 20s 225ms/step - accuracy: 0.6476 - loss: 0.5975 - val_accuracy: 0.4375 - val_loss: 1.0945
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 19s 215ms/step - accuracy: 0.6497 - loss: 0.5787 - val_accuracy: 0.4319 - val_loss: 1.1690
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 20s 218ms/step - accuracy: 0.6510 - loss: 0.5793 - val_accuracy: 0.4347 - val_loss: 1.0740
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 20s 220ms/step - accuracy: 0.6608 - loss: 0.5690 - val_accuracy: 0.4319 - val_loss: 1.4450
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 19s 215ms/step - accuracy: 0.6611 - loss: 0.5654 - val_accuracy: 0.4306 - val_loss: 1.3186
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 21s 238ms/step - accuracy: 0.6670 - loss: 0.5515 - val_accuracy: 0.4347 - val_loss: 1.4624
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 19s 211ms/step - accuracy: 0.6722 - loss: 0.5478 - val_accu